# 02 - Counterfactual Consistency Evaluation (item 6)

Runs the full harness from `counterfactual.py` against the current rule-based `verify()`. See that module's docstring for the perturbation categories and why flip rate is conditioned on `original_verdict == 'supported'` for the breaking category.

In [ ]:
# == Environment + paths =========================================================
import json, os, re, sys, time
from pathlib import Path

# Kaggle detection uses an env var Kaggle actually sets, not path existence: on
# Windows, Path('/kaggle/working').exists() is NOT a reliable off-Kaggle check --
# pathlib treats a leading '/' as relative to the current drive's root, so an
# unrelated local C:\kaggle or F:\kaggle directory would silently return True.
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))

def _find(name, prefer_largest=True):
    """Search for `name`; if multiple matches exist, prefer the largest file --
    a complete/authoritative cache is virtually always bigger than a stale or
    partial one left over from an earlier run."""
    roots = ["/kaggle/input"] if ON_KAGGLE else ["..", "../..", "."]
    hits = []
    for root in roots:
        r = Path(root)
        if r.exists():
            hits.extend(r.rglob(name))
    if not hits:
        return None
    if prefer_largest:
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
    return hits[0]

def _find_src_dir():
    """Locate the directory holding this project's shared modules (identified by
    containing graph_utils.py, not by a fixed relative path) -- on Kaggle, an
    attached dataset lands under /kaggle/input/<dataset-name>/..., so neither
    '../src' nor 'src' exists there; search by content the same way _find() above
    locates data files, instead of assuming a fixed layout."""
    for c in (Path("../src"), Path("src"), Path("/kaggle/working/src")):
        if (c / "graph_utils.py").exists():
            return c
    if ON_KAGGLE:
        hits = list(Path("/kaggle/input").rglob("graph_utils.py"))
        if hits:
            return hits[0].parent
    return Path("src")  # last resort; will raise ModuleNotFoundError on import if wrong

SRC_DIR = _find_src_dir()
sys.path.insert(0, str(SRC_DIR))

BENCH = _find("scireqa_full_relational_v1.1_humanmerged.jsonl") or _find("scireqa_full_relational_v1.1.jsonl")
REFS_CACHE = _find("citecheck_refs_cache.json")
CITE_RESULTS = _find("citecheck_results.json")
BENCH_VERIFIED = _find("scireqa_human_verified_subset.jsonl")
assert BENCH and REFS_CACHE and CITE_RESULTS, "attach the SciRecBench + CiteCheck output dataset"

OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ON_KAGGLE:", ON_KAGGLE)
print("SRC_DIR:", SRC_DIR.resolve() if SRC_DIR.exists() else f"{SRC_DIR} (NOT FOUND -- attach the src/ dataset)")
print("BENCH:", BENCH, f"({BENCH.stat().st_size:,} bytes)" if BENCH else "")
print("REFS_CACHE:", REFS_CACHE, f"({REFS_CACHE.stat().st_size:,} bytes)" if REFS_CACHE else "")
print("CITE_RESULTS:", CITE_RESULTS)
print("BENCH_VERIFIED:", BENCH_VERIFIED)
print("OUT_DIR:", OUT_DIR)


In [ ]:
from graph_utils import CitationGraph, load_relational_claims
from counterfactual import run_counterfactual_suite, summarize, BREAKING_PERTURBATIONS
from graph_utils import DEPENDENCY_RELATIONS
import json

graph = CitationGraph.load(REFS_CACHE)
claims = load_relational_claims(BENCH, CITE_RESULTS)
claims = [c for c in claims if c["recorded_verdict"] is not None]
print(f"Loaded {len(claims)} relational claims, graph has {len(graph)} nodes.")

results = run_counterfactual_suite(claims, graph, seed=0)
print(f"Ran {len(results)} (claim, perturbation) trials.\n")

summary = summarize(results)
print(f"{'perturbation':26s} {'category':12s} {'n scored':>9s} {'/total':>7s} {'flip rate':>10s}")
for pert, s in summary.items():
    print(f"{pert:26s} {s['category']:12s} {s['n_scored']:9d} {s['n_total_trials']:7d} {s['flip_rate']:10.1%}")

# breakdown by dependency vs non-dependency for the breaking perturbations
from collections import defaultdict
by = defaultdict(lambda: [0, 0])
for r in results:
    if r.perturbation in BREAKING_PERTURBATIONS and r.original_verdict == "supported":
        cat = "dependency" if r.relation in DEPENDENCY_RELATIONS else "non_dependency"
        key = f"{r.perturbation}__{cat}"
        by[key][0] += 1
        by[key][1] += int(r.changed)
by_relation_category = {k: {"n": n, "flip_rate": f / n} for k, (n, f) in by.items()}
for k, v in sorted(by_relation_category.items()):
    print(f"{k:40s} n={v['n']:5d} flip_rate={v['flip_rate']:.1%}")

with open(OUT_DIR / "counterfactual_results.json", "w", encoding="utf-8") as f:
    json.dump({"overall_summary": summary, "by_relation_category": by_relation_category,
               "n_claims": len(claims), "n_trials": len(results)}, f, indent=2)
print("\nSaved to", OUT_DIR / "counterfactual_results.json")
